Databricks notebook source
SENSE 프로젝트 — Silver Layer: yfinance raw → silver
담당: 1조 (정형 데이터)

목적:
  ADLS Gen2 raw/yfinance/ 에서 1년치 주가 데이터를 읽어
  master_calendar 기반 시계열 정제 및 파생 컬럼 생성 후 silver/yfinance/ 에 저장

처리 대상 티커:
  [KR] 005930.KS (삼성전자), 000660.KS (SK하이닉스)
  [US] NVDA, TSM, AMD, INTC, ASML, MU, WDC, ^SOX

[파생 컬럼]
  log_return        : 로그 수익률 ln(Pt / Pt-1)
  volatility_gk     : Garman-Klass 일별 변동성
  volatility_5d     : 5일 실현 변동성 ← Gold 타겟 변수 기반
  effective_kr_date : master_calendar 기반 한국 유효 영업일

[master_calendar 활용]
  - 타임존 시프트: 미국 종목 +1일 후 한국 실제 영업일로 정확히 보정
  - 유효 행 필터: 한국 휴장일 행 제거 (KR 종목)
  - Forward Fill: 미국 공휴일 결측치 보간


# 0. 스토리지 계정 설정 및 ADLS OAuth 인증


In [0]:
# ============================================================
# 스토리지 계정 및 경로 상수 정의
# ============================================================
STORAGE_ACCOUNT  = "3dtteam1adls"
SECRET_SCOPE     = "sense-kv"
BRONZE_CONTAINER = "raw"
SILVER_CONTAINER = "curated"

BASE_PATH_BRONZE    = f"abfss://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
BASE_PATH_SILVER = f"abfss://{SILVER_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# master_calendar 경로 (raw 컨테이너 내 공용 리소스)
CALENDAR_PATH = f"{BASE_PATH_SILVER}/master_calendar.parquet"

# ============================================================
# ADLS Gen2 OAuth 인증 (Service Principal)
# ============================================================
spark.conf.set(
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    dbutils.secrets.get(scope=SECRET_SCOPE, key="adls-client-id")
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    dbutils.secrets.get(scope=SECRET_SCOPE, key="adls-client-secret")
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{dbutils.secrets.get(scope=SECRET_SCOPE, key='adls-tenant-id')}/oauth2/token"
)

print(f"✅ ADLS OAuth 인증 설정 완료: {STORAGE_ACCOUNT}")


# 1. master_calendar 로드

팀원이 제작한 한국·미국 영업일 캘린더를 로드합니다.

| 컬럼 | 설명 |
|---|---|
| `기준일자` | 날짜 (2024-01-01 ~ 2026-04-10) |
| `주말여부` | 토·일 여부 |
| `한국_휴장일_여부` | 주말 + 한국 공휴일 |
| `미국_휴장일_여부` | 주말 + 미국 공휴일 |

> **활용 포인트**
> - 한국만 휴장: 34건 (설날, 추석, 광복절 등)
> - 미국만 휴장: 18건 (MLK Day, 독립기념일 등)
> - 이 불일치를 단순 주말 보정으로는 잡을 수 없습니다.


In [0]:
from pyspark.sql.functions import to_date, col as F_col
import pyspark.sql.functions as F

# master_calendar 로드
calendar_df = (
    spark.read
    .parquet(CALENDAR_PATH)
    .withColumn("기준일자", to_date(F_col("기준일자"), "yyyy-MM-dd"))
    .select("기준일자", "주말여부", "한국_휴장일_여부", "미국_휴장일_여부")
)

# 한국 실제 영업일 (주말 X, 한국 공휴일 X)
kr_biz_days = calendar_df.filter(F_col("한국_휴장일_여부") == False).select(
    F_col("기준일자").alias("kr_biz_date")
)

total      = calendar_df.count()
kr_biz_cnt = kr_biz_days.count()
us_biz_cnt = calendar_df.filter(F_col("미국_휴장일_여부") == False).count()

print(f"✅ master_calendar 로드 완료")
print(f"   전체 기간  : {total}일")
print(f"   한국 영업일: {kr_biz_cnt}일")
print(f"   미국 영업일: {us_biz_cnt}일")
print(f"   한국만 휴장: {calendar_df.filter((F_col('한국_휴장일_여부')==True) & (F_col('미국_휴장일_여부')==False)).count()}건")
print(f"   미국만 휴장: {calendar_df.filter((F_col('미국_휴장일_여부')==True) & (F_col('한국_휴장일_여부')==False)).count()}건")
display(calendar_df.limit(10))


# 2. yfinance raw 데이터 로드 (1년치 전체)

raw 경로 구조:
```
raw/yfinance/
  year=2025/month=01/day=01/yahoo_finance_raw_20250101.csv
  year=2026/month=04/day=08/yahoo_finance_raw_20260408.csv
```
`recursiveFileLookup=true` 로 하위 파티션 전체를 한번에 로드합니다.


In [0]:
from pyspark.sql.functions import to_date, year, month, dayofmonth, col

yfinance_raw_경로 = f"{BASE_PATH_RAW}/yfinance"

yfinance_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("recursiveFileLookup", "true")
    .option("encoding", "utf-8")
    .option("inferSchema", "true")
    .load(yfinance_raw_경로)
)

print(f"✅ yfinance raw 로드 완료: {yfinance_df.count()}행")
print(f"   컬럼: {yfinance_df.columns}")
display(yfinance_df.limit(5))


# 3. 스키마 정리

- `Date` → `date` (DateType)
- 파티션 컬럼 `year`, `month`, `day` 추가
- 중복 제거: (`date`, `Ticker`) 기준
- 필수값 null 제거: `Close`, `date`


In [0]:
yfinance_clean_df = (
    yfinance_df
    .withColumn("date",  to_date(col("Date"), "yyyy-MM-dd"))
    .withColumn("year",  year(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("day",   dayofmonth(col("date")))
    .dropDuplicates(["date", "Ticker"])
    .filter(col("date").isNotNull())
    .filter(col("Close").isNotNull())
)

print(f"✅ 스키마 정리 완료: {yfinance_clean_df.count()}행")
yfinance_clean_df.printSchema()
display(yfinance_clean_df.limit(5))

# 4. 미국 / 한국 티커 분리 및 타임존 시프트 (master_calendar 기반)

### 기존 방식의 문제
단순 +1일 후 주말만 보정 → 미국·한국 공휴일 불일치 51건 미처리

### 새 방식
미국 종목: `date + 1일` 이후 **한국 실제 영업일** 중 가장 가까운 날로 보정
한국 종목: **한국 휴장일 행 자체를 제거** (Forward Fill 대신 원천 차단)

> 예시: 2024-09-16~18 (추석) 한국 휴장
> → 미국 NVDA 데이터는 2024-09-19(목)으로 시프트
> → 기존 방식은 2024-09-17(화)로 잘못 매핑됨


In [0]:
import pyspark.sql.functions as F
from pyspark.sql import Window

US_TICKERS = ["NVDA", "TSM", "AMD", "INTC", "ASML", "MU", "WDC", "^SOX"]
KR_TICKERS = ["005930.KS", "000660.KS"]

us_df = yfinance_clean_df.filter(F.col("Ticker").isin(US_TICKERS))
kr_df = yfinance_clean_df.filter(F.col("Ticker").isin(KR_TICKERS))

# ── 미국 종목: date+1일 → 한국 실제 영업일로 보정 ──────────────────────
# 한국 영업일 캘린더에 row_number 부여 → 날짜 기준 다음 영업일 찾기
kr_biz_indexed = kr_biz_days.withColumn(
    "rn", F.row_number().over(Window.orderBy("kr_biz_date"))
)

# US 종목에 +1일 적용
us_plus1 = us_df.withColumn("shifted_date", F.date_add(F.col("date"), 1))

# shifted_date 이후 첫 번째 한국 영업일 찾기 (asof join 패턴)
# 방법: kr_biz_days와 cross 후 shifted_date <= kr_biz_date 인 최솟값
us_shifted_df = (
    us_plus1
    .join(
        kr_biz_indexed,
        us_plus1["shifted_date"] <= kr_biz_indexed["kr_biz_date"],
        "left"
    )
    .groupBy(
        *[c for c in us_plus1.columns]
    )
    .agg(F.min("kr_biz_date").alias("effective_kr_date"))
)

# ── 한국 종목: 한국 휴장일 행 제거 ────────────────────────────────────────
kr_shifted_df = (
    kr_df
    .join(
        calendar_df.filter(F.col("한국_휴장일_여부") == False)
                   .select(F.col("기준일자").alias("date")),
        on="date",
        how="inner"   # 휴장일은 JOIN 안 되므로 자동 제거
    )
    .withColumn("effective_kr_date", F.col("date"))
)

print("✅ 타임존 시프트 완료 (master_calendar 기반)")
print(f"   미국 종목 행수: {us_shifted_df.count():,}")
print(f"   한국 종목 행수: {kr_shifted_df.count():,}")
display(
    us_shifted_df
    .select("Ticker", "date", "effective_kr_date", "Close")
    .orderBy("Ticker", "date")
    .limit(10)
)


# 5. 파생 컬럼 생성

| 컬럼명 | 수식 | 용도 |
|---|---|---|
| `log_return` | ln(Ct / Ct-1) | 수익률 계산 |
| `volatility_gk` | Garman-Klass 추정량 | 정밀 일별 변동성 |
| `volatility_5d` | √(mean(log_return²) rolling 5d) | **Gold 타겟 변수 기반** |

Window 기준: `Ticker` 파티션 + `effective_kr_date` 정렬
> 캘린더 기반 정제 후 계산하므로 휴장일로 인한 날짜 왜곡 없음


In [0]:
import math
from pyspark.sql import Window
from pyspark.sql.types import DoubleType

def add_derived_columns(df):
    """log_return, volatility_gk, volatility_5d 파생 컬럼 추가"""
    w_ticker = Window.partitionBy("Ticker").orderBy("effective_kr_date")
    w_5d     = w_ticker.rowsBetween(-4, 0)

    return (
        df
        # 로그 수익률
        .withColumn(
            "log_return",
            F.log(F.col("Close") / F.lag("Close", 1).over(w_ticker))
        )
        # Garman-Klass 변동성
        # σ²_GK = 0.5·ln(H/L)² - (2ln2-1)·ln(C/O)²
        .withColumn(
            "volatility_gk",
            F.when(
                (F.col("High") > 0) & (F.col("Low") > 0) & (F.col("Open") > 0),
                F.sqrt(
                    F.lit(0.5) * F.pow(F.log(F.col("High") / F.col("Low")), 2)
                    - F.lit(2.0 * math.log(2) - 1.0)
                      * F.pow(F.log(F.col("Close") / F.col("Open")), 2)
                )
            ).otherwise(F.lit(None).cast(DoubleType()))
        )
        # 5일 실현 변동성
        .withColumn(
            "volatility_5d",
            F.sqrt(F.avg(F.pow(F.col("log_return"), 2)).over(w_5d))
        )
        # 첫 행 null → 0 채우기 (lag/rolling 윈도우 초기값 부재)
        .na.fill(0.0, subset=["log_return", "volatility_5d"])
    )

us_featured_df = add_derived_columns(us_shifted_df)
kr_featured_df = add_derived_columns(kr_shifted_df)

print("✅ 파생 컬럼 생성 완료")
display(
    us_featured_df
    .select("Ticker", "date", "effective_kr_date", "Close",
            "log_return", "volatility_gk", "volatility_5d")
    .orderBy("Ticker", "effective_kr_date")
    .limit(10)
)

   
# 6. 통합 및 데이터 검증

### 공휴일 검증 로직
검증 시 조인 키는 `date`가 아닌 **`effective_kr_date`** 를 사용합니다.

| 컬럼 | 의미 | 한국 공휴일 포함 가능? |
|---|---|---|
| `date` | 원래 거래일 (미국 기준) | ⭕ 미국 시장은 한국 공휴일에도 거래 |
| `effective_kr_date` | 한국 유효 영업일로 시프트된 날짜 | ❌ 항상 한국 영업일 |

> `date` 기준으로 검증하면 미국 종목이 한국 공휴일에 거래한 정상 행이
> 오탐(false positive)으로 잡힙니다. (8티커 × 14일 = 112건)

In [0]:
# US + KR 통합
unified_df = us_featured_df.drop("shifted_date").unionByName(kr_featured_df)

# ── 티커별 행수 ──────────────────────────────────────────────────────
print("=== 📊 티커별 행수 ===")
display(unified_df.groupBy("Ticker").count().orderBy("Ticker"))

# ── 날짜 범위 ────────────────────────────────────────────────────────
date_range = unified_df.agg(
    F.min("date").alias("start_date"),
    F.max("date").alias("end_date"),
    F.countDistinct("date").alias("영업일수")
).collect()[0]

print(f"\n=== 📅 날짜 범위 ===")
print(f"  시작: {date_range['start_date']}")
print(f"  종료: {date_range['end_date']}")
print(f"  영업일수: {date_range['영업일수']}일  (1년 기준 약 250일)")

# ── volatility_5d null 비율 ──────────────────────────────────────────
total      = unified_df.count()
null_cnt   = unified_df.filter(F.col("volatility_5d").isNull()).count()
ticker_cnt = unified_df.select("Ticker").distinct().count()

print(f"\n=== 🔍 volatility_5d null 비율 ===")
print(f"  전체: {total:,}행  |  null: {null_cnt:,}행  |  비율: {null_cnt/total*100:.1f}%")
print(f"  (정상 범위: 티커수({ticker_cnt}) × 4 = {ticker_cnt * 4}행)")

# ── 캘린더 기반 검증: effective_kr_date가 한국 휴장일에 걸리지 않는지 확인 ──
print("\n=== 📆 한국 휴장일 잔존 여부 확인 (effective_kr_date 기준) ===")
kr_holidays = calendar_df.filter(
    (F.col("한국_휴장일_여부") == True) & (F.col("주말여부") == False)
).select(F.col("기준일자").alias("effective_kr_date"))

leak_cnt = unified_df.join(kr_holidays, on="effective_kr_date", how="inner").count()
if leak_cnt == 0:
    print("  ✅ 한국 공휴일 행 없음 — 정상")
else:
    print(f"  ⚠️ 한국 공휴일 행 {leak_cnt}건 잔존 — 확인 필요")

# 7. Silver 저장

- 포맷: **Parquet** (기존 노트북 방식 유지)
- 파티션: `year` / `month`
- 모드: `overwrite`

Silver 저장 시 `overwrite` (전체 덮어쓰기) 를 사용하는 이유는
**파생 컬럼이 과거 데이터를 참조하기 때문**입니다.

- `volatility_5d` = 최근 5일치 `log_return` 기반 계산
- 신규 데이터만 `append` 하면 경계 구간에서 오계산 발생 가능
- 현재 규모(약 5,000행)에서 전체 재처리 비용 = 수십 초 수준으로 무시 가능

> 데이터 규모가 커지면 Delta Table + Watermark 증분 방식으로 전환 고려


In [0]:
SILVER_PATH = f"{BASE_PATH_SILVER}/yfinance"

(
    unified_df
    .write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet(SILVER_PATH)
)

# 저장 확인
saved_files = dbutils.fs.ls(SILVER_PATH)
print(f"✅ Silver 저장 완료")
print(f"   경로     : {SILVER_PATH}")
print(f"   파티션 수: {len(saved_files)}개")
for f in saved_files:
    print(f"   - {f.name}")


# 8. 저장 확인 (sanity check)


In [0]:
df_check = spark.read.parquet(SILVER_PATH)

print(f"✅ silver 읽기 확인 — 행수: {df_check.count():,}")
print(f"   컬럼: {df_check.columns}")
display(df_check.orderBy("Ticker", "effective_kr_date").limit(10))


## ✅ 완료 후 다음 단계

| 순서 | 노트북 | 상태 |
|---|---|---|
| 1 | `01_yfinance_raw_to_silver.py` | ✅ 완료 |
| 2 | `02_fred_raw_to_silver.py` | ⏳ 대기 |
| 3 | `03_fx_raw_to_silver.py` | ⏳ 대기 |
| 4 | `04_silver_to_gold.py` | ⏳ 대기 |

> **master_calendar 경로 확인:**
> `raw/master_calendar/master_calendar.parquet`
> 실제 ADLS 업로드 경로가 다르면 `CALENDAR_PATH` 변수 수정 후 실행하세요.
